# Streamlit interface

The local interface displays conversation and exposes the route, retrieval evidence, MCP tool arguments and results, token counts and timing. The final answer is displayed when complete, while TTFT is measured from the streamed response. The retrieved recipes are synthetic examples and the interface makes their limitations visible.


## Runtime code
Run this cell from the v2 directory to write the corresponding runnable file.


In [ ]:
%%writefile app.py
"""Run with: streamlit run v2/app.py from the project folder."""
import os
import sys
import time
from pathlib import Path
import streamlit as st
from dotenv import load_dotenv

ROOT = Path(__file__).resolve().parent
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT.parent / '.env')
from chef.retrieval import build_index
from chef.agent import create_graph

st.set_page_config(page_title='Flavor Chef V2', page_icon='🍲')
st.title('🍲 Flavor Chef')
st.caption('Version 2 · recipes, dietary questions and deterministic kitchen calculations')
st.info('The recipe collection contains synthetic teaching examples. Cooking times are estimates; verify allergy information independently.')
if not os.getenv('GROQ_API_KEY') or os.getenv('GROQ_API_KEY') == 'your_key_here':
    st.warning('Add GROQ_API_KEY to NLPproject/.env to use the assistant.')
    st.stop()

@st.cache_resource(show_spinner='Loading the recipe index and assistant…')
def get_graph(strategy, top_k):
    return create_graph(build_index(strategy), top_k=top_k)

with st.sidebar:
    st.header('Retrieval')
    strategy = st.selectbox('Chunking strategy', ['sections', 'recipe'], help='Compare focused sections with complete recipes.')
    top_k = st.slider('Retrieved fragments', 1, 10, 5)
    if st.button('Clear conversation'):
        st.session_state.messages = []
        st.rerun()
    st.caption('Token counts are provider-reported. Time to first token refers to the final answer model call.')

if 'messages' not in st.session_state: st.session_state.messages = []
for message in st.session_state.messages:
    with st.chat_message(message['role']):
        st.markdown(message['content'])
        if 'details' in message:
            with st.expander('Sources, tools and measurements'):
                st.json(message['details'])
question = st.chat_input('Ask about a recipe or calculate a quantity')
if question:
    history = [{'role': m['role'], 'content': m['content']} for m in st.session_state.messages[-8:]]
    st.session_state.messages.append({'role': 'user', 'content': question})
    with st.chat_message('user'): st.markdown(question)
    with st.chat_message('assistant'):
        try:
            started = time.perf_counter()
            result = get_graph(strategy, top_k).invoke({'question': question, 'history': history})
            elapsed = round(time.perf_counter() - started, 3)
            st.markdown(result['answer'])
            calls = result.get('calls', [])
            def total(field):
                values = [call.get(field) for call in calls]
                return sum(values) if values and all(v is not None for v in values) else None
            details = {'route':result.get('route'), 'retrieval_query':result.get('search_query'), 'retrieved_chunks':result.get('context', []), 'mcp_call':result.get('tool_call'), 'mcp_result':result.get('tool_result'), 'model_calls':calls, 'total_input_tokens':total('input_tokens'), 'total_output_tokens':total('output_tokens'), 'answer_ttft_seconds':result.get('ttft_seconds'), 'total_seconds':elapsed, 'other_timings':result.get('timings', {})}
            with st.expander('Sources, tools and measurements'): st.json(details)
            st.session_state.messages.append({'role':'assistant','content':result['answer'],'details':details})
        except Exception as exc:
            st.error(f'Request failed: {type(exc).__name__}: {exc}')
